In [98]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder, OrdinalEncoder)
pd.set_option('display.max_columns', 1000)


In [99]:
df=pd.read_csv("/home/salah-mohammed/Data_Science/ds_jobs_project/data/cleaned_dfs/updated_df.csv")
df.head()

,Job Title,Company,Location,Salary Estimate,Job Description,Job Age,Country,Hourly,Glassdoor_Estimate,Employer_Provided,currency,min_salary,max_salary,avg_salary,Skills,Web development,ERP systems,Supply chain,Research,Pro/ENGINEER,SAS,ServiceNow,Biostatistics,Database management,MRP,.NET Core,Content development,NoSQL,Machine learning,Sensors,Web analytics,Recruiting,IT,CAD,Oracle,SaaS,Microsoft Powerpoint,Virtualization,Yardi,Databases,Microsoft Windows Server,Logistics,Root cause analysis,Authentication,Scripting,Sales,Data entry,Filing,SharePoint,Clinical laboratory experience,Multilingual,Tooling,HIPAA,Software deployment,DoD experience,Statistical software,Ansible,Program management,Social listening,Unreal Engine,Waterfall,FPGA,Robotics,Organizational skills,Cloud computing,Taxonomy,Laboratory information management systems,Computer skills,Conflict management,Vendor management,Usability,FEA,Help desk,SSO,Algorithm design,Customer retention,Management,Data science,Redshift,Microsoft Word,SOX,Google Analytics,Financial modeling,Operating systems,Scientific research,Natural language processing,Process improvement,Kubernetes,Business analysis,Computer networking,Internet of Things,Construction,Optics,Relationship management,Figma,Scalability,Google Cloud Platform,Presentation skills,S3,Test-driven development,Data analysis skills,Biotechnology,Cloud infrastructure,Application development,Information security,XML,Cassandra,Ontology,GitHub,Signal processing,SSRS,IIS,Talent acquisition,Terraform,French,Microsoft SQL Server,Mandarin,MATLAB,Project management software,Distributed systems,Administrative experience,WAN,C,R,VoIP,Big data,Requirements gathering,Bootstrap,Fanuc,System design,Investment,Data centre experience,D3.js,Systems engineering,Program design,Agile,A/B testing,Statistical analysis,Microsoft PowerPoint,Embedded software,GAMP,Git,Marketing mix modeling,FedRAMP,Wealth management,AI,Telecommunication,Statistics,Configuration management,Disaster recovery,Managed care,SAP,Financial planning,Quantitative analysis,Computer operation,Performance tuning,SolidWorks,TCP,Mechanical engineering,SDKs,Pricing,Load balancing,Spark,Web accessibility,English,Warehouse management system,Cloud development,Public health,FISMA,Salesforce Marketing Cloud,React,Accounting software,UI,T-SQL,Proposal writing,New Relic,Controlling experience,Cucumber (software testing tool),Trello,Qualitative research,Cost management,Hadoop,Forecasting,Experimental design,Accounts payable,Python,Revenue cycle management,Node.js,Analytics,EDI,Kanban,Geometry,TensorFlow,Research & development,Data warehouse,Metadata,Image processing,Change management,Project coordination,Performance management,Banking,Salt,Microsoft Office,Bilingual,Data collection,Moving,Spanish,CSS,User acceptance testing,Computer vision,Quality assurance,E-commerce,Maths,Public speaking,Contract management,Data analytics,Requirements analysis,Rust (programming language),Barista experience,Clinical development,Product development,Docker,Salesforce,Schematics,NetSuite,Software architecture,Military,Leadership,Jira,Design patterns,Data visualization,Visual Basic,Data mining,Deep learning,Communication skills,Quality control,JavaScript,MongoDB,Data modeling,Operations research,Epic,Project leadership,Alteryx,Team management,Operations management,Customer service,Software troubleshooting,Internet of things,Terrafrom,Clinical research,Software testing,Business requirements,Medical imaging,Adobe Photoshop,Japanese,Marketing,C#,Data structures,Linux,Microservices,Market analysis,DB2,Hospitality,Regression analysis,Wireframing,Enterprise software,APIs,Writing skills,Russian,Java,Tableau,Market research,Incident management,Microsoft Excel,Microsoft Access,Continuous improvement,Graph databases,Electrical engineering,Test automation,.NET,Selenium,QlikView,Revit,5G,Financial services,Confluence,AWS,JSON,Azure,Project management,Data management,Systems design,Computer literacy,HP ALM,Power BI,Google S

### EDA Functions

In [100]:
"""UNIVARIATE PLOTTING FUNCTIONS FOR EDA"""
# Add the print statements to the function
def explore_categorical(df, x, fillna = True, placeholder = 'MISSING',
                        figsize = (6,4), order = None):
  """Creates a seaborn countplot with the option to temporarily fill missing values
  Prints statements about null values, cardinality, and checks for
  constant/quasi-constant features.
  Source:{PASTE IN FINAL LESSON LINK}
  """
  # Make a copy of the dataframe and fillna
  temp_df = df.copy()
  # Before filling nulls, save null value counts and percent for printing
  null_count = temp_df[x].isna().sum()
  null_perc = null_count/len(temp_df)* 100
  # fillna with placeholder
  if fillna == True:
    temp_df[x] = temp_df[x].fillna(placeholder)
  # Create figure with desired figsize
  fig, ax = plt.subplots(figsize=figsize)
  # Plotting a count plot
  sns.countplot(data=temp_df, x=x, ax=ax, order=order)
  # Rotate Tick Labels for long names
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
  # Add a title with the feature name included
  ax.set_title(f"Column: {x}", fontweight='bold')

  # Fix layout and show plot (before print statements)
  fig.tight_layout()
  plt.show()
  print("*****"*10)
  # Print null value info
  print(f"- NaN's Found: {null_count} ({round(null_perc,2)}%)")
  # Print cardinality info
  nunique = temp_df[x].nunique()
  print(f"- Unique Values: {nunique}")
  # First find value counts of feature
  val_counts = temp_df[x].value_counts(dropna=False)
  # Define the most common value
  most_common_val = val_counts.index[0]
  # Define the frequency of the most common value
  freq = val_counts.values[0]
  # Calculate the percentage of the most common value
  perc_most_common = freq / len(temp_df) * 100
  # Print the results
  print(f"- Most common value: '{most_common_val}' occurs {freq} times ({round(perc_most_common,2)}%)")
  # print message if quasi-constant or constant (most common val more than 98% of data)
  if perc_most_common > 98:
    print(f"\n- [!] Warning: '{x}' is a constant or quasi-constant feature and should be dropped.")
  else:
    print("- Not constant or quasi-constant.")
  return fig, ax


# TO DO: add the new print statements from explore_categorical
def explore_numeric(df, x, figsize=(6,5) ):
  """Creates a seaborn histplot and boxplot with a share x-axis,
  Prints statements about null values, cardinality, and checks for
  constant/quasi-constant features.
  Source:{PASTE IN FINAL LESSON LINK}
  """

  ## Save null value counts and percent for printing
  null_count = df[x].isna().sum()
  null_perc = null_count/len(df)* 100


  ## Making our figure with gridspec for subplots
  gridspec = {'height_ratios':[0.7,0.3]}
  fig, axes = plt.subplots(nrows=2, figsize=figsize,
                           sharex=True, gridspec_kw=gridspec)
  # Histogram on Top
  sns.histplot(data=df, x=x, ax=axes[0])

  # Boxplot on Bottom
  sns.boxplot(data=df, x=x, ax=axes[1])

  ## Adding a title
  axes[0].set_title(f"Column: {x}", fontweight='bold')

  ## Adjusting subplots to best fill Figure
  fig.tight_layout()

  # Ensure plot is shown before message
  plt.show()

  print("*****"*10)
  # Print null value info
  print(f"- NaN's Found: {null_count} ({round(null_perc,2)}%)")
  # Print cardinality info
  nunique = df[x].nunique()
  print(f"- Unique Values: {nunique}")


  # Get the most most common value, its count as # and as %
  most_common_val_count = df[x].value_counts(dropna=False).head(1)
  most_common_val = most_common_val_count.index[0]
  freq = most_common_val_count.values[0]
  perc_most_common = freq / len(df) * 100

  print(f"- Most common value: '{most_common_val}' occurs {freq} times ({round(perc_most_common,2)}%)")

  # print message if quasi-constant or constant (most common val more than 98% of data)
  if perc_most_common > 98:
    print(f"\n- [!] Warning: '{x}' is a constant or quasi-constant feature and should be dropped.")
  else:
    print("- Not constant or quasi-constant.")
  return fig, axes


In [101]:
"""MULTIVARIATE PLOTTING FUNCTIONS VS. NUMERIC TARGET"""

def plot_categorical_vs_target(df, x, y='rating',figsize=(6,4),
                            fillna = True, placeholder = 'MISSING',
                            order = None):
  """Plots a combination of a seaborn barplot of means combined with
  a seaborn stripplot to show the spread of the data.
  Source:{PASTE IN FINAL LESSON LINK}
  """
  # Make a copy of the dataframe and fillna
  temp_df = df.copy()
  # fillna with placeholder
  if fillna == True:
    temp_df[x] = temp_df[x].fillna(placeholder)

  # or drop nulls prevent unwanted 'nan' group in stripplot
  else:
    temp_df = temp_df.dropna(subset=[x])
  # Create the figure and subplots
  fig, ax = plt.subplots(figsize=figsize)

    # Barplot
  sns.barplot(data=temp_df, x=x, y=y, ax=ax, order=order, alpha=0.6,
              linewidth=1, edgecolor='black', errorbar=None)

  # Boxplot
  sns.stripplot(data=temp_df, x=x, y=y, hue=x, ax=ax,
                order=order, hue_order=order, legend=False,
                edgecolor='white', linewidth=0.5,
                size=3,zorder=0)
  # Rotate xlabels
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

  # Add a title
  ax.set_title(f"{x} vs. {y}", fontweight='bold')
  fig.tight_layout()
  return fig, ax


def plot_numeric_vs_target(df, x, y,
                           figsize=(6,4),
                           ):
  """Plots a seaborn regplot with Pearson's correlation (r) added
  to the title.
  Source:{PASTE IN FINAL LESSON LINK}
  """
  # Calculate the correlation
  corr = df[[x,y]].corr().round(2)
  r = corr.loc[x,y]

  # Plot the data
  fig, ax = plt.subplots(figsize=figsize)
  scatter_kws={'ec':'white','alpha':0.8}
  sns.regplot(data=df, x=x, y=y, ax=ax,scatter_kws=scatter_kws)

  ## Add the title with the correlation
  ax.set_title(f"{x} vs. {y} (r = {r})", fontweight='bold')

  # Make sure the plot is shown before the print statement
  plt.show()

  return fig, ax

# + Cont'd Data Preperation phase

In [102]:
drop_cols=[ 'Job Title','Company', 'Location', 'Salary Estimate', "Job Description", 'Job Age' ,'Country','Skills','min_salary','max_salary','avg_salary','currency' ]
# drop title, company, location, job descreption, job age,salary estimate , and skills are basis str (reg model wont learn, you need nlp)
# min_salary, max_salary to avoid data leakage (since target is avg_salary)
# job age text , and even we made it int, wont be that helpful
# currency captured by country, and avg_salary is in unified currency ($)

In [103]:
# drop target nans always , dont do fillna, or even preproccessing, to avoid leakage, and to combine  fake data with real data in target
# dont train model with nan target valus
df_model=df.dropna(subset='avg_salary').copy() 



In [104]:
# df_model.isna().sum()
# df_model.loc[df_model.isna().any(axis=1)]
# # rest nans are in Skills col, X so in preprocessing 


In [105]:
##! split data
X=df_model.drop(columns=drop_cols)
y=df_model['avg_salary']
X_train, X_test, y_train, y_test= train_test_split(X,y, test_size=0.2,random_state=42)
X_train.head()

,Hourly,Glassdoor_Estimate,Employer_Provided,Web development,ERP systems,Supply chain,Research,Pro/ENGINEER,SAS,ServiceNow,Biostatistics,Database management,MRP,.NET Core,Content development,NoSQL,Machine learning,Sensors,Web analytics,Recruiting,IT,CAD,Oracle,SaaS,Microsoft Powerpoint,Virtualization,Yardi,Databases,Microsoft Windows Server,Logistics,Root cause analysis,Authentication,Scripting,Sales,Data entry,Filing,SharePoint,Clinical laboratory experience,Multilingual,Tooling,HIPAA,Software deployment,DoD experience,Statistical software,Ansible,Program management,Social listening,Unreal Engine,Waterfall,FPGA,Robotics,Organizational skills,Cloud computing,Taxonomy,Laboratory information management systems,Computer skills,Conflict management,Vendor management,Usability,FEA,Help desk,SSO,Algorithm design,Customer retention,Management,Data science,Redshift,Microsoft Word,SOX,Google Analytics,Financial modeling,Operating systems,Scientific research,Natural language processing,Process improvement,Kubernetes,Business analysis,Computer networking,Internet of Things,Construction,Optics,Relationship management,Figma,Scalability,Google Cloud Platform,Presentation skills,S3,Test-driven development,Data analysis skills,Biotechnology,Cloud infrastructure,Application development,Information security,XML,Cassandra,Ontology,GitHub,Signal processing,SSRS,IIS,Talent acquisition,Terraform,French,Microsoft SQL Server,Mandarin,MATLAB,Project management software,Distributed systems,Administrative experience,WAN,C,R,VoIP,Big data,Requirements gathering,Bootstrap,Fanuc,System design,Investment,Data centre experience,D3.js,Systems engineering,Program design,Agile,A/B testing,Statistical analysis,Microsoft PowerPoint,Embedded software,GAMP,Git,Marketing mix modeling,FedRAMP,Wealth management,AI,Telecommunication,Statistics,Configuration management,Disaster recovery,Managed care,SAP,Financial planning,Quantitative analysis,Computer operation,Performance tuning,SolidWorks,TCP,Mechanical engineering,SDKs,Pricing,Load balancing,Spark,Web accessibility,English,Warehouse management system,Cloud development,Public health,FISMA,Salesforce Marketing Cloud,React,Accounting software,UI,T-SQL,Proposal writing,New Relic,Controlling experience,Cucumber (software testing tool),Trello,Qualitative research,Cost management,Hadoop,Forecasting,Experimental design,Accounts payable,Python,Revenue cycle management,Node.js,Analytics,EDI,Kanban,Geometry,TensorFlow,Research & development,Data warehouse,Metadata,Image processing,Change management,Project coordination,Performance management,Banking,Salt,Microsoft Office,Bilingual,Data collection,Moving,Spanish,CSS,User acceptance testing,Computer vision,Quality assurance,E-commerce,Maths,Public speaking,Contract management,Data analytics,Requirements analysis,Rust (programming language),Barista experience,Clinical development,Product development,Docker,Salesforce,Schematics,NetSuite,Software architecture,Military,Leadership,Jira,Design patterns,Data visualization,Visual Basic,Data mining,Deep learning,Communication skills,Quality control,JavaScript,MongoDB,Data modeling,Operations research,Epic,Project leadership,Alteryx,Team management,Operations management,Customer service,Software troubleshooting,Internet of things,Terrafrom,Clinical research,Software testing,Business requirements,Medical imaging,Adobe Photoshop,Japanese,Marketing,C#,Data structures,Linux,Microservices,Market analysis,DB2,Hospitality,Regression analysis,Wireframing,Enterprise software,APIs,Writing skills,Russian,Java,Tableau,Market research,Incident management,Microsoft Excel,Microsoft Access,Continuous improvement,Graph databases,Electrical engineering,Test automation,.NET,Selenium,QlikView,Revit,5G,Financial services,Confluence,AWS,JSON,Azure,Project management,Data management,Systems design,Computer literacy,HP ALM,Power BI,Google Suite,Interviewing,Django,Release management,Calculus,Test management tools,Business intelligence,Analysis skills,Windows,Vis

In [106]:
X_train.isna().sum().sum() # good luck that there is no nans, but still we will add imputer (i just want )

np.int64(0)

## Preproccessing

In [107]:
cat_cols=X_train.select_dtypes('str').columns

impute_mode=SimpleImputer(strategy="most_frequent")
ohe=OneHotEncoder(handle_unknown='ignore',sparse_output=False)

cat_pipe=make_pipeline(impute_mode,ohe)
cat_tuple=['Categorical',cat_pipe,cat_cols]

##! ****************************************************
num_cols=X_train.select_dtypes('number').columns

impute_median=SimpleImputer(strategy='median')
scaler=StandardScaler()

num_pipe=make_pipeline(impute_median,scaler)
num_tuple=['Numarical',num_pipe,num_cols]


In [ ]:
# remmeber until now we didnt do any fit, in modeling we will fit with the preprocessor and model at the same time
preprocessor=ColumnTransformer(transformers=[cat_tuple,num_tuple],verbose_feature_names_out=False)
preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[['Categorical', Pipeline(step...tput=False))]), ...], ['Numarical', Pipeline(step...ardScaler())]), ...]]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be form

# Modeling

In [111]:
from sklearn.linear_model import LinearRegression
# let's start with dummy model to see the flow
dummy_lin_reg = LinearRegression()
dummy_lin_reg_pipe=make_pipeline(preprocessor,dummy_lin_reg)
dummy_lin_reg_pipe.fit(X_train,y_train)
dummy_lin_reg_pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('linearregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[['Categorical', Pipeline(step...tput=False))]), ...], ['Numarical', Pipeline(step...ardScaler())]), ...]]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"spars

In [ ]:
print(f"the intercept for dummy linear reg model is {dummy_lin_reg.intercept_}")
# 111 is the avg_salary for a default job with no skills
print(f"the coefecient for dummy linear reg model is {dummy_lin_reg.coef_}")


the intercept for dummy linear reg model is 111.8165401991065
the coefecient for dummy linear reg model is [-4.33878756e+00 -3.06376705e+00  7.40255460e+00  3.01738488e+01
 -2.82694650e+01 -2.78351623e-01 -1.00119447e+01  8.38591252e+00
  5.86569720e-01 -1.00507552e+01  1.00507552e+01  2.12982074e+00
 -1.11698105e+00  1.34188678e+00  1.79887764e-01  0.00000000e+00
 -2.11206103e+00  9.24296226e-01  1.05588455e+01 -3.46389584e-14
  2.93098879e-14  1.03064504e+01 -5.07198941e-01  1.44153145e+00
 -7.69622684e+00 -5.15143483e-14  2.88657986e-14  1.05884969e+00
 -3.39591187e+00 -3.15908346e-01 -3.73921679e+00  5.32907052e-15
  1.55054982e+00 -1.28352694e+00 -6.21724894e-15 -5.47601832e+00
  1.13458116e+00 -2.10502772e-01  2.14138069e+00 -1.77635684e-15
  5.67568346e-01  2.95004384e-02 -1.07986211e+00 -4.44089210e-15
  6.63452019e+00  6.26559660e-01  7.51280377e+00 -1.81167238e-01
  1.70226011e+00  4.34267385e+00  3.54864219e+00  3.19744231e-14
 -2.08958862e+00  5.67396420e+00 -2.88902049e-01

In [114]:
dummy_lin_reg_pipe.get_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[['Categorical',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='most_frequent')),
                                                    ('onehotencoder',
                                                     OneHotEncoder(handle_unknown='ignore',
                                                                   sparse_output=False))]),
                                    Index(['seniority', 'Job Category'], dtype='str')],
                                   ['Numarical',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='median')),
                                                    ('standardscaler',
                                                     Stan...,
                                    Index(['Hourly',

In [116]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error ,root_mean_squared_error

def regression_metrics(y_true, y_pred, label='', verbose = True, output_dict=False):
  # Get metrics
  mae = mean_absolute_error(y_true, y_pred)
  mse = mean_squared_error(y_true, y_pred)
  rmse = root_mean_squared_error(y_true, y_pred)
  r_squared = r2_score(y_true, y_pred)
  if verbose == True:
    # Print Result with Label and Header
    header = "-"*60
    print(header, f"Regression Metrics: {label}", header, sep='\n')
    print(f"- MAE = {mae:,.3f}")
    print(f"- MSE = {mse:,.3f}")
    print(f"- RMSE = {rmse:,.3f}")
    print(f"- R^2 = {r_squared:,.3f}")
  if output_dict == True:
      metrics = {'Label':label, 'MAE':mae,
                 'MSE':mse, 'RMSE':rmse, 'R^2':r_squared}
      return metrics

def evaluate_regression(reg, X_train, y_train, X_test, y_test, verbose = True,
                        output_frame=False):
  # Get predictions for training data
  y_train_pred = reg.predict(X_train)

  # Call the helper function to obtain regression metrics for training data
  results_train = regression_metrics(y_train, y_train_pred, verbose = verbose,
                                     output_dict=output_frame,
                                     label='Training Data')
  print()
  # Get predictions for test data
  y_test_pred = reg.predict(X_test)
  # Call the helper function to obtain regression metrics for test data
  results_test = regression_metrics(y_test, y_test_pred, verbose = verbose,
                                  output_dict=output_frame,
                                    label='Test Data' )

  # Store results in a dataframe if ouput_frame is True
  if output_frame:
    results_df = pd.DataFrame([results_train,results_test])
    # Set the label as the index
    results_df = results_df.set_index('Label')
    # Set index.name to none to get a cleaner looking result
    results_df.index.name=None
    # Return the dataframe
    return results_df.round(3)

In [117]:
evaluate_regression(dummy_lin_reg_pipe,X_train,y_train,X_test,y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 16.787
- MSE = 844.932
- RMSE = 29.068
- R^2 = 0.780

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 47.382
- MSE = 4,604.336
- RMSE = 67.855
- R^2 = -0.424


In [119]:
from sklearn.model_selection import GridSearchCV

lin_reg_param_grid = {
    'linearregression__fit_intercept': [True, False],
    'linearregression__positive': [True, False]
}
grid_search_lin_reg=GridSearchCV(estimator= dummy_lin_reg_pipe,param_grid=lin_reg_param_grid,cv=5,scoring='neg_mean_squared_error' )
grid_search_lin_reg.fit(X_train,y_train)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'linearregression__fit_intercept': [True, False], 'linearregression__positive': [True, False]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candi

In [120]:
evaluate_regression(grid_search_lin_reg,X_train,y_train,X_test,y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 22.357
- MSE = 1,295.757
- RMSE = 35.997
- R^2 = 0.663

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 36.902
- MSE = 2,476.964
- RMSE = 49.769
- R^2 = 0.234
